# 📓 Notebook 2 — Fine-tuning XLM-RoBERTa
**Project:** Indonesian Government Law Sentiment Analysis  
**Model:** `cardiffnlp/twitter-xlm-roberta-base-sentiment` → fine-tuned for 6 emotions  
**Dataset:** EmoTweetID-Human (2,243 tweets, 6 emotion classes)

---
**Pipeline:**
```
Load splits → Tokenize → Freeze base layers → Train classifier → Evaluate → Save model
```

> ⚠️ **Before running:** Make sure you enabled GPU in Colab
> Runtime → Change runtime type → T4 GPU

## ⚙️ Step 1 — Install Dependencies

In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy seaborn matplotlib -q
print('✅ Packages ready')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

# --- Device check ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — training will be very slow. Enable GPU in Runtime settings.')

## 📂 Step 2 — Load Splits & Label Mappings

> Upload the files output from Notebook 1:
> `split_train.csv`, `split_val.csv`, `split_test.csv`, `class_weights.json`, `label_mapping.json`

In [ ]:
# --- Load data splits ---
train_df = pd.read_csv('split_train.csv')
val_df   = pd.read_csv('split_val.csv')
test_df  = pd.read_csv('split_test.csv')

# --- Load label mapping ---
with open('label_mapping.json', 'r') as f:
    mapping = json.load(f)
LABEL2ID = mapping['label2id']
ID2LABEL = {int(k): v for k, v in mapping['id2label'].items()}

# --- Load class weights ---
with open('class_weights.json', 'r') as f:
    weights_dict = json.load(f)
class_weights = torch.tensor(
    [weights_dict[str(i)] for i in range(len(LABEL2ID))],
    dtype=torch.float
).to(device)

print(f'✅ Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'✅ Labels: {list(LABEL2ID.keys())}')
print(f'✅ Class weights loaded: {[round(w.item(),3) for w in class_weights]}')

## 🔤 Step 3 — Tokenizer & Model Config

In [ ]:
# ============================================================
# Base model — twitter-aware XLM-RoBERTa (multilingual tweets)
# We fine-tune this further for 6 emotion classes
# ============================================================
MODEL_NAME  = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
MAX_LEN     = 128   # Twitter max is ~280 chars, 128 tokens covers most
NUM_LABELS  = len(LABEL2ID)  # 6

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded')

# --- Test tokenizer ---
sample = 'gak setuju sama UU media sosial anak ini'
tokens = tokenizer(sample, return_tensors='pt')
print(f'Sample token count: {tokens["input_ids"].shape[1]}')

## 🗂️ Step 4 — PyTorch Dataset Class

In [ ]:
class TweetEmotionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.texts  = dataframe['tweet_clean'].astype(str).values
        self.labels = dataframe['label_id'].values
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ============================================================
# HYPERPARAMETERS — adjust if you want to experiment
# ============================================================
BATCH_SIZE    = 16    # reduce to 8 if GPU OOM error
EPOCHS        = 5     # early stopping will kick in if val loss plateaus
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1   # 10% of steps for warmup
FREEZE_LAYERS = 8     # freeze bottom N transformer layers (out of 12)
                      # increase to 10 if overfitting, decrease to 6 if underfitting
PATIENCE      = 2     # early stopping patience (epochs)
# ============================================================

train_dataset = TweetEmotionDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = TweetEmotionDataset(val_df,   tokenizer, MAX_LEN)
test_dataset  = TweetEmotionDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'✅ Datasets ready')
print(f'   Train batches : {len(train_loader)}')
print(f'   Val batches   : {len(val_loader)}')
print(f'   Test batches  : {len(test_loader)}')

## 🧊 Step 5 — Load Model & Freeze Base Layers

We freeze the bottom 8 transformer layers and only train the top 4 + classifier head.  
This prevents overfitting on our small dataset (2,243 samples).

In [ ]:
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True  # needed since we change num_labels
)
model = model.to(device)
print('✅ Model loaded')

# --- Freeze bottom FREEZE_LAYERS transformer layers ---
frozen = 0
for name, param in model.named_parameters():
    # Freeze embeddings
    if 'embeddings' in name:
        param.requires_grad = False
        frozen += 1
    # Freeze bottom N encoder layers
    elif 'encoder.layer' in name:
        layer_num = int(name.split('encoder.layer.')[1].split('.')[0])
        if layer_num < FREEZE_LAYERS:
            param.requires_grad = False
            frozen += 1

# Count trainable vs frozen
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n🧊 Frozen layers: bottom {FREEZE_LAYERS} of 12 transformer layers')
print(f'   Total params     : {total_params:,}')
print(f'   Trainable params : {trainable_params:,} ({trainable_params/total_params*100:.1f}%)')
print(f'   Frozen params    : {total_params - trainable_params:,} ({(total_params-trainable_params)/total_params*100:.1f}%)')

## 🏋️ Step 6 — Training Setup

In [ ]:
# --- Weighted loss (handles class imbalance) ---
criterion = nn.CrossEntropyLoss(weight=class_weights)

# --- Optimizer (only updates trainable params) ---
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

# --- Learning rate scheduler with warmup ---
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'✅ Training setup ready')
print(f'   Total steps  : {total_steps}')
print(f'   Warmup steps : {warmup_steps}')
print(f'   Epochs       : {EPOCHS}')
print(f'   Batch size   : {BATCH_SIZE}')
print(f'   Learning rate: {LEARNING_RATE}')

## 🚀 Step 7 — Training Loop with Early Stopping

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds    = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), correct / total, f1


# --- Training loop ---
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_val_f1   = 0
patience_ctr  = 0
SAVE_PATH     = 'xlmroberta_emotion_best.pt'

print('🚀 Starting training...\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"Val F1":>6}')
print('-' * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    flag = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), SAVE_PATH)
        patience_ctr = 0
        flag = ' ✅ best'
    else:
        patience_ctr += 1
        flag = f' (patience {patience_ctr}/{PATIENCE})'

    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>8.4f} | {val_acc:>7.2%} | {val_f1:>6.4f}{flag}')

    if patience_ctr >= PATIENCE:
        print(f'\n⏹️  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print(f'\n🏆 Best Val F1: {best_val_f1:.4f} — model saved to {SAVE_PATH}')

## 📈 Step 8 — Training Curves

In [ ]:
epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(x, history['train_loss'], 'o-', label='Train', color='#3498db')
axes[0].plot(x, history['val_loss'],   's-', label='Val',   color='#e74c3c')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend()

# Accuracy
axes[1].plot(x, history['train_acc'], 'o-', label='Train', color='#3498db')
axes[1].plot(x, history['val_acc'],   's-', label='Val',   color='#e74c3c')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Val F1
axes[2].plot(x, history['val_f1'], 'D-', color='#27ae60')
axes[2].set_title('Validation Macro F1', fontweight='bold')
axes[2].set_xlabel('Epoch')
best_epoch = history['val_f1'].index(max(history['val_f1'])) + 1
axes[2].axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch {best_epoch}')
axes[2].legend()

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

## 🧪 Step 9 — Final Evaluation on Test Set

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()
print(f'✅ Loaded best model from {SAVE_PATH}')

# Run on test set
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        preds   = outputs.logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]

print('\n=== TEST SET RESULTS ===')
print(f'Accuracy : {accuracy_score(all_labels, all_preds):.4f}')
print(f'Macro F1 : {f1_score(all_labels, all_preds, average="macro"):.4f}')
print()
print('=== PER-CLASS REPORT ===')
print(classification_report(all_labels, all_preds, target_names=label_names))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Percentages
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (% of true class)', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrix saved')
print()
print('💡 TIP: Check which emotions are most confused with each other.')
print('   Sadness/fear and anger/disgust pairs are typically hardest to separate.')

## 💾 Step 10 — Save Full Model for Notebook 3

In [ ]:
# Save full model + tokenizer (Hugging Face format — easiest to reload)
EXPORT_DIR = 'xlmroberta_emotion_finetuned'
os.makedirs(EXPORT_DIR, exist_ok=True)

model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Save label mapping alongside model
with open(f'{EXPORT_DIR}/label_mapping.json', 'w') as f:
    json.dump({'label2id': LABEL2ID, 'id2label': {str(k): v for k, v in ID2LABEL.items()}}, f)

print(f'✅ Model saved to: {EXPORT_DIR}/')
print(f'   Contains: config.json, model.safetensors, tokenizer files, label_mapping.json')

# Zip for easy download
!zip -r xlmroberta_emotion_finetuned.zip xlmroberta_emotion_finetuned/
print('✅ Zipped: xlmroberta_emotion_finetuned.zip')

In [ ]:
# --- Optional: Download model + charts ---
from google.colab import files

print('📥 Downloading outputs...')
files.download('xlmroberta_emotion_finetuned.zip')
files.download('training_curves.png')
files.download('confusion_matrix.png')

## 🔬 Step 11 — Quick Inference Test (Sanity Check)

In [ ]:
def predict_emotion(texts, model, tokenizer, device, max_len=128):
    """Predict emotion for a list of texts. Returns label + confidence."""
    model.eval()
    results = []

    encoding = tokenizer(
        texts,
        max_length=max_len,
        padding=True,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        outputs = model(**encoding)
        probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()

    for i, text in enumerate(texts):
        pred_id    = probs[i].argmax()
        pred_label = ID2LABEL[pred_id]
        confidence = probs[i][pred_id]
        all_scores = {ID2LABEL[j]: round(float(probs[i][j]), 3) for j in range(NUM_LABELS)}
        results.append({
            'text':       text,
            'emotion':    pred_label,
            'confidence': round(float(confidence), 3),
            'all_scores': all_scores
        })
    return results


# --- Test with sample tweets about the law ---
test_samples = [
    "gak setuju banget sama UU ini, tidak masuk akal sama sekali",
    "Alhamdulillah akhirnya pemerintah peduli sama keselamatan anak di medsos",
    "takut juga sih kalau anak kecil bebas pakai sosmed tanpa pengawasan",
    "jijik lihat politisi yang pura-pura peduli anak padahal cuma cari panggung",
    "sedih banget lihat anak SD udah kecanduan TikTok seharian",
    "heran kenapa baru sekarang pemerintah bikin aturan ini",
]

predictions = predict_emotion(test_samples, model, tokenizer, device)

print('=== SANITY CHECK — Sample Predictions ===')
for p in predictions:
    print(f'\nText      : {p["text"]}')
    print(f'Predicted : {p["emotion"].upper()} (confidence: {p["confidence"]:.1%})')
    scores_str = ' | '.join([f'{k}: {v:.2f}' for k, v in sorted(p['all_scores'].items(), key=lambda x: -x[1])])
    print(f'Scores    : {scores_str}')

---
## ✅ Notebook 2 Complete!

| Output | Used in |
|---|---|
| `xlmroberta_emotion_finetuned/` | Notebook 3 — inference |
| `xlmroberta_emotion_finetuned.zip` | Download & backup |
| `training_curves.png` | Report / thesis |
| `confusion_matrix.png` | Report / thesis |

**What to check before moving on:**
- Macro F1 ideally > 0.60 for 6-class small dataset — if lower, try reducing `FREEZE_LAYERS` to 6
- If `surprise` F1 is very low (< 0.40), that's expected — it's the most ambiguous emotion
- If val loss goes up while train loss goes down → overfitting → increase `FREEZE_LAYERS` to 10

**➡️ Next: Notebook 3 — Inference on scraped tweets + visualization**